# RQ3: What has a greater impact on final performance: changing the pre-training technique or the encoder architecture?

## Define base configs

In [ ]:
import pandas as pd
metric_used = 'accuracy'
apply_correction_factor = True


def extract_metric_target(df):
    metric_map = {
        "har": "accuracy",
        "hapt": "accuracy",
    }
    return df["pipeline/task"].map(metric_map)


def extract_metric(df):
    metric_results = []
    metric_map = {
        "har": "metric/classification/accuracy",
        "hapt": "metric/classification/accuracy",
    }

    for _, row in df.iterrows():
        metric_column = metric_map[row["pipeline/task"]]
        metric_value = row[metric_column]
        metric_results.append(metric_value)

    return metric_results

In [ ]:
from pathlib import Path


from utils import (
    calculate_variant_wilcoxon,
    create_precedence_graph,
    prepare_experiment_df,
    plot_comparison_bar
)

# disable warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Path to the experiment results (parsed) summarized_executions_fixed
filename = 'clean_saved_metrics_paper_final'
summarized_executions_path = Path(f"{filename}.csv")

# summarized_executions_path = Path("summarized_executions_fixed.csv")

# Location to save figures and tables
figures_path = Path("results/figures/")
tables_path = Path("results/tables/")
figures_path.mkdir(parents=True, exist_ok=True)
tables_path.mkdir(parents=True, exist_ok=True)
print(f"Sucessfully created directories '{figures_path}' and '{tables_path}'")

## Utility functions

First, we define some utility functions that will be used to parse the benchmarks results and to generate the plots.

In [ ]:
def result_pairwise_wilcoxon(
    df, variants_variables, filters={}, show_stats=False,apply_bonferroni= False,
):
    df = prepare_experiment_df(
        df,
        **filters,
        verbose=True,
    )
    # display(df)
    if show_stats:
        print("Dataframe has", len(df.index), "rows")
        for col in df.columns:
            values = df[col].unique()
            print(f" - {col} ({len(values)})", values)

    return calculate_variant_wilcoxon(df, variants_variables, threshold=0.05,apply_bonferroni= apply_bonferroni)


def result_precedence_graph(
    df, variants_variables, filters={}, show_stdev=False,apply_bonferroni= False,show_df_tests=False
):
    w_df = result_pairwise_wilcoxon(df, variants_variables, filters,apply_bonferroni= apply_bonferroni,)
    if show_df_tests:
        print("Pairwise Wilcoxon test results:")
        display(w_df)
    return create_precedence_graph(w_df, show_stdev=show_stdev,apply_bonferroni= apply_bonferroni),w_df


def show_precedence_graph(
    df, variants_variables, filters, filename_suffix=None, show_stdev=False,apply_bonferroni= False,show_df_tests=False,filename=None
):

    dot,w_df = result_precedence_graph(
        df=df,
        variants_variables=variants_variables,
        filters=filters,
        show_stdev=show_stdev,
        apply_bonferroni=apply_bonferroni,
        show_df_tests=show_df_tests,
    )
    if filename is None:
        
        filename = "precedence_graph-" + "-".join(variants_variables)
    if filename_suffix:
        filename = filename + "-" + filename_suffix
    dot.render(
        filename=filename, directory=figures_path, format="png", cleanup=True
    )
    print(f"Precedence graph saved to '{figures_path / filename}.png'\n")
    display(dot)
    return dot,w_df


def summarize_backbone_performance(df):
    # Get all unique backbones (from both Variant 1 and Variant 2)
    all_backbones = set(df["Variant 1"]).union(set(df["Variant 2"]))
    
    # Initialize a dictionary to store stats for each backbone
    backbone_stats = {}
    
    for backbone in all_backbones:
        # Get all rows where the backbone appears (either in Variant 1 or Variant 2)
        mask = (df["Variant 1"] == backbone) | (df["Variant 2"] == backbone)
        relevant_rows = df[mask]
        
        # Extract means and stds where the backbone is involved
        means = []
        stds = []
        
        for _, row in relevant_rows.iterrows():
            if row["Variant 1"] == backbone:
                means.append(row["Variant 1 Mean"])
                stds.append(row["Variant 1 stdev"])
            else:
                means.append(row["Variant 2 Mean"])
                stds.append(row["Variant 2 stdev"])
        
        # Compute mean and std across all occurrences
        mean_performance = np.mean(means) if means else 0
        avg_std = np.mean(stds) if stds else 0
        
        backbone_stats[backbone] = {
            "Mean": mean_performance,
            "Std": avg_std,
            "Mean ± Std": f"{np.round(mean_performance*100, 1)}% ± {np.round(avg_std*100, 1)}%"
        }
    # --- Original logic for wins/losses ---
    sig_df = df[df["Significant"] == True].copy()
    
    # Determine winner and loser based on means
    sig_df["Winner"] = sig_df.apply(
        lambda row: row["Variant 1"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 2"], 
        axis=1
    )
    sig_df["Loser"] = sig_df.apply(
        lambda row: row["Variant 2"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 1"], 
        axis=1
    )
    
    # Count wins and losses
    win_counts = sig_df["Winner"].value_counts()
    loss_counts = sig_df["Loser"].value_counts()
    
    # Create summary DataFrame
    summary_df = pd.DataFrame.from_dict(backbone_stats, orient="index")
    summary_df.index.name = "Backbone"
    summary_df.reset_index(inplace=True)
    
    # Ensure all backbones are included (even if no wins/losses)
    summary_df["Wins"] = summary_df["Backbone"].map(win_counts).fillna(0).astype(int)
    summary_df["Losses"] = summary_df["Backbone"].map(loss_counts).fillna(0).astype(int)
    summary_df["Net Score"] = summary_df["Wins"] - summary_df["Losses"]
    
    # Sort by Net Score (descending)
    summary_df.sort_values(["Net Score", "Mean"], ascending=[False, False], inplace=True)
    
    return summary_df


In [ ]:
def aggregate_backbone_performance(combined_df):
    """
    Processes a combined DataFrame of backbone results to produce:
    1. Technique-specific performance (mean ± std)
    2. Backbone totals across all techniques
    
    Args:
        combined_df: DataFrame containing results from multiple techniques
        
    Returns:
        tuple: (technique_summary_df, backbone_totals_df)
    """
    # --- Technique-Specific Summary ---
    technique_summary = combined_df.copy()
    
    # Convert to percentages if needed (assuming original means are 0-1)
    technique_summary["Mean"] = technique_summary["Mean"] * 100
    technique_summary["Std"] = technique_summary["Std"] * 100
    
    # Format performance string
    technique_summary["Performance"] = (
        technique_summary["Mean"].round(1).astype(str) + 
        "% ± " + 
        technique_summary["Std"].round(1).astype(str) + "%"
    )
    
    # Sort by Net Score then Mean
    # technique_summary = technique_summary.sort_values(
    #     ["Net Score", "Mean"], 
    #     ascending=[False, False]
    # )
    technique_summary = technique_summary.sort_values(
        ["Mean","Net Score"], 
        ascending=[False, False]
    )
    
    # --- Backbone Totals ---
    # Extract backbone name (before " + ")
    combined_df["Backbone Only"] = combined_df["Backbone"].str.split(" \+ ").str[0]
    
    # Group and aggregate
    backbone_totals = combined_df.groupby("Backbone Only").agg({
        "Wins": "sum",
        "Losses": "sum",
        "Net Score": "sum",
        "Mean": lambda x: np.mean(x) * 100,  # Convert to percentage
        "Std": lambda x: np.mean(x) * 100
    }).reset_index()
    
    # Format performance
    backbone_totals["Performance"] = (
        backbone_totals["Mean"].round(1).astype(str) + 
        "% ± " + 
        backbone_totals["Std"].round(1).astype(str) + "%"
    )
    
    # Clean up
    backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
    # backbone_totals = backbone_totals.sort_values(
    #     ["Net Score", "Mean"], 
    #     ascending=[False, False]
    # )
    backbone_totals = backbone_totals.sort_values(
        ["Mean","Net Score"], 
        ascending=[False, False]
    )
    # Special handling for TS2Vec if present
    if "TS2Vec" in backbone_totals["Backbone"].values:
        backbone_totals["Backbone"] = backbone_totals["Backbone"].replace({
            "TS2Vec": "TS2Vec (Partial)"
        })
    
    # Select final columns
    backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]
    
    return technique_summary, backbone_totals

def generate_performance_tables(df_list, technique_names=None):
    """
    Complete workflow from individual technique DataFrames to final tables
    
    Args:
        df_list: List of DataFrames for each technique
        technique_names: Optional list of technique names
        
    Returns:
        tuple: (combined_df, technique_summary, backbone_totals)
    """
    # Add technique identifiers if provided
    if technique_names and len(technique_names) == len(df_list):
        for df, name in zip(df_list, technique_names):
            df["Technique"] = name
    
    # Combine all DataFrames
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Generate summary tables
    technique_summary, backbone_totals = aggregate_backbone_performance(combined_df)
    
    return combined_df, technique_summary, backbone_totals



In [ ]:
summarized_executions_path

In [ ]:
df = pd.read_csv(summarized_executions_path)
df


# mudar técnica ou backbone?

O que traz mais benefícios: mudar a estratégia de SSL dado um backbone e um dataset para pré-treino, ou mudar o backbone (mantendo a mesma técnica de SSL)? 

In [ ]:
base_df = df.copy()

In [ ]:
import numpy as np

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["TNC"],#,"Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
# display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TNC",
]

# Generate all tables
combined_df, technique_summary_tnc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tnc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# TFC

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["TFC"],#,"Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
# display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "TFC",
]

# Generate all tables
combined_df, technique_summary_tfc, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_tfc)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# Diet

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["Diet"],#,"Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
# display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "Diet",
]

# Generate all tables
combined_df, technique_summary_diet, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_diet)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# LFR

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["LFR"],#,"Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
# display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "LFR",
]

# Generate all tables
combined_df, technique_summary_lfr, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_lfr)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# supervised

# freeze by dataset
df = base_df
datasets = ['UCI', 'RW-Thigh', 'RW-Waist', 'MS', 'WISDM', 'KH']
# fractions = ["100.0%", "50.0%", "10.0%", "5.0%", "1.0%"]

results_by_dataset = []

for dataset in datasets:
    # for frac in fractions:
    
    dot, df_variant = show_precedence_graph(
        df,
        variants_variables=["backbone", "d_target"],
        filters={
            "select_d_target": [dataset],
            # "select_frac_dtarget": [frac],
            # "select_backbones": ["ResNet","RNN","Transformer","CNN"],
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": ["Supervised"],#,"Diet","Supervised","TFC"],
            # Não filtramos tsk_pretext → considera todos (TFC, Diet, LFR, Supervised...)
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
    )
    summary_df = summarize_backbone_performance(df_variant)
    summary_df["Dataset"] = dataset
    # summary_df["Data Fraction"] = frac
    results_by_dataset.append(summary_df)


# Combina todos os resultados
combined_df_rq4_by_dataset = pd.concat(results_by_dataset, ignore_index=True)
# display(combined_df_rq4_by_dataset)

# Resumo por backbone (total geral)
summary_df_rq4_by_dataset = (
    combined_df_rq4_by_dataset
    .groupby("Backbone")
    .agg({"Wins": "sum", "Losses": "sum"})
    .reset_index()
)
summary_df_rq4_by_dataset["Net Score"] = summary_df_rq4_by_dataset["Wins"] - summary_df_rq4_by_dataset["Losses"]
summary_df_rq4_by_dataset = summary_df_rq4_by_dataset.sort_values("Net Score", ascending=False)

# Exibe resumo final
# display(summary_df_rq4_by_dataset)

# Separa apenas o nome do backbone (antes do " + ")
combined_df_rq4_by_dataset["Backbone Only"] = combined_df_rq4_by_dataset["Backbone"].str.split(" \+ ").str[0]
technique_dfs = [
    combined_df_rq4_by_dataset
]

technique_names = [
    "Supervised",
]

# Generate all tables
combined_df, technique_summary_supervised, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary_supervised)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
combined_df = pd.concat([
    technique_summary_supervised,
    technique_summary_tfc,
    technique_summary_lfr,
    technique_summary_tnc,
    technique_summary_diet,
])


# combined_df['Dataset Name'] = combined_df['Dataset'].str.split(r'\s+\+\s+').str[0]
# Exemplo: 'CNN + UCI + 100.0%' → 'CNN'
combined_df['Backbone'] = combined_df['Backbone'].str.extract(r'^(.+?)\s+\+')


combined_df

# analise completa

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy import stats
import plotly.express as px

# plt.style.use('seaborn')
sns.set_palette("husl")

def comprehensive_analysis(df):
    # 1. Filtro e preparação dos dados
    valid_techniques = ['Supervised', 'TFC', 'LFR', 'TNC', 'Diet']
    valid_backbones = ['CNN-PFF', 'TS2Vec Encoder', 'RNN', 'ResNet-1D',"ResNet-SE-5", 'IMU Transformer','TS-TCC Encoder']
    
    df = df.copy()
    df = df[df['Technique'].isin(valid_techniques)]
    df = df[df['Backbone'].isin(valid_backbones)]
    
    # 2. Análise de todas as combinações de técnicas
    tech_comparison = []
    
    for (dataset, backbone), group in df.groupby(['Dataset', 'Backbone']):
        tech_means = group.groupby('Technique')['Mean'].mean()
        if len(tech_means) >= 2:
            for tech1, tech2 in combinations(tech_means.index, 2):
                delta = tech_means[tech2] - tech_means[tech1]
                tech_comparison.append({
                    'Dataset': dataset,
                    'Backbone': backbone,
                    'Tech1': tech1,
                    'Tech2': tech2,
                    'Delta': delta,
                    'Tech1_Mean': tech_means[tech1],
                    'Tech2_Mean': tech_means[tech2],
                    'Comparison': f"{tech1} vs {tech2}"
                })
    
    tech_comp_df = pd.DataFrame(tech_comparison)
    
    # 3. Análise de todas as combinações de backbones
    bb_comparison = []
    
    for (dataset, technique), group in df.groupby(['Dataset', 'Technique']):
        bb_means = group.groupby('Backbone')['Mean'].mean()
        if len(bb_means) >= 2:
            for bb1, bb2 in combinations(bb_means.index, 2):
                delta = bb_means[bb2] - bb_means[bb1]
                bb_comparison.append({
                    'Dataset': dataset,
                    'Technique': technique,
                    'BB1': bb1,
                    'BB2': bb2,
                    'Delta': delta,
                    'BB1_Mean': bb_means[bb1],
                    'BB2_Mean': bb_means[bb2],
                    'Comparison': f"{bb1} vs {bb2}"
                })
    
    bb_comp_df = pd.DataFrame(bb_comparison)
    
    # 4. Cálculo de métricas agregadas
    tech_stats = tech_comp_df.groupby('Comparison')['Delta'].agg(['mean', 'std', 'count'])
    bb_stats = bb_comp_df.groupby('Comparison')['Delta'].agg(['mean', 'std', 'count'])
    
    # 5. Visualizações
    plt.figure(figsize=(18, 12))
    
    # Heatmap de comparação de técnicas
    plt.subplot(2, 2, 1)
    tech_pivot = tech_comp_df.pivot_table(index=['Backbone', 'Dataset'], 
                                         columns='Comparison', values='Delta')
    sns.heatmap(tech_pivot, cmap="coolwarm", center=0, annot=True, fmt=".1f")
    plt.title('Ganho de Performance entre Técnicas por Backbone/Dataset')
    plt.xticks(rotation=45, ha='right')
    
    # Distribuição dos ganhos por técnica
    plt.subplot(2, 2, 2)
    sns.boxplot(data=tech_comp_df, x='Delta', y='Comparison', showfliers=False)
    plt.axvline(0, color='red', linestyle='--')
    plt.title('Distribuição dos Ganhos entre Técnicas')
    plt.xlabel('Ganho de Performance (%)')
    
    # Heatmap de comparação de backbones
    plt.subplot(2, 2, 3)
    bb_pivot = bb_comp_df.pivot_table(index=['Technique', 'Dataset'], 
                                     columns='Comparison', values='Delta')
    sns.heatmap(bb_pivot, cmap="coolwarm", center=0, annot=True, fmt=".1f")
    plt.title('Ganho de Performance entre Backbones por Técnica/Dataset')
    plt.xticks(rotation=45, ha='right')
    
    # Distribuição dos ganhos por backbone
    plt.subplot(2, 2, 4)
    sns.boxplot(data=bb_comp_df, x='Delta', y='Comparison', showfliers=False)
    plt.axvline(0, color='red', linestyle='--')
    plt.title('Distribuição dos Ganhos entre Backbones')
    plt.xlabel('Ganho de Performance (%)')
    
    plt.tight_layout()
    plt.show()

    
    return {
        'tech_comparisons': tech_comp_df,
        'bb_comparisons': bb_comp_df,
        'tech_stats': tech_stats,
        'bb_stats': bb_stats
    }

# Executar a análise completa
results = comprehensive_analysis(combined_df)

In [ ]:
tech_comp_df, bb_comp_df, tech_stats, bb_stats = (
    results['tech_comparisons'],
    results['bb_comparisons'],
    results['tech_stats'],
    results['bb_stats']
)


In [ ]:
tech_stats

In [ ]:
tech_stats = tech_stats.reset_index().rename(columns={'index': 'Comparison'})

# Function to process each row
def adjust_row(row):
    if row['mean'] < 0:
        a, b = row['Comparison'].split(' vs ')
        row['Comparison'] = f"{b} → {a}"
        row['mean'] = abs(row['mean'])
    else:
        row['Comparison'] = row['Comparison'].replace(' vs ', ' → ')
    return row

# Apply the transformation
tech_stats = tech_stats.apply(adjust_row, axis=1)
tech_stats = tech_stats.sort_values(by='mean',ascending=False)
tech_stats['mean'] = tech_stats['mean'].round(2)
tech_stats['std'] = tech_stats['std'].round(2)
display(tech_stats)

mean_of_means = tech_stats['mean'].mean().round(2)
mean_of_stds = tech_stats['std'].mean().round(2)

print(f"Mean of means: {mean_of_means}")
print(f"Mean of stds: {mean_of_stds}")


In [ ]:
bb_stats = bb_stats.reset_index().rename(columns={'index': 'Comparison'})

display(bb_stats)

# Apply the transformation
bb_stats = bb_stats.apply(adjust_row, axis=1)
bb_stats = bb_stats.sort_values(by='mean',ascending=False)
bb_stats['mean'] = bb_stats['mean'].round(2)
bb_stats['std'] = bb_stats['std'].round(2)
display(bb_stats)
mean_of_means = bb_stats['mean'].mean().round(2)
mean_of_stds = bb_stats['std'].mean().round(2)

print(f"Mean of means: {mean_of_means}")
print(f"Mean of stds: {mean_of_stds}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy import stats
import plotly.express as px

# Seaborn styling for publication
sns.set(style="whitegrid", font_scale=1.5)
sns.set_context(
    "paper",
    rc={
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.labelweight": "bold",
        "grid.color": "gray",
        "grid.linestyle": "--",
        "grid.linewidth": 1.0,
    },
)

def comprehensive_analysis(df):
    # 1. Data filtering and preparation
    valid_techniques = ['Supervised', 'TFC', 'LFR', 'TNC', 'Diet']
    valid_backbones = ['CNN-PFF', 'TS2Vec Encoder', 'RNN', 'ResNet-1D',"ResNet-SE-5", 'IMU Transformer','TS-TCC Encoder']
    
    df = df.copy()
    df = df[df['Technique'].isin(valid_techniques)]
    df = df[df['Backbone'].isin(valid_backbones)]
    
    # 2. Analysis of all technique combinations
    tech_comparison = []
    
    for (dataset, backbone), group in df.groupby(['Dataset', 'Backbone']):
        tech_means = group.groupby('Technique')['Mean'].mean()
        if len(tech_means) >= 2:
            for tech1, tech2 in combinations(tech_means.index, 2):
                delta = tech_means[tech2] - tech_means[tech1]
                tech_comparison.append({
                    'Dataset': dataset,
                    'Backbone': backbone,
                    'Tech1': tech1,
                    'Tech2': tech2,
                    'Delta': delta,
                    'Tech1_Mean': tech_means[tech1],
                    'Tech2_Mean': tech_means[tech2],
                    'Comparison': f"{tech1} vs {tech2}"
                })
    
    tech_comp_df = pd.DataFrame(tech_comparison)
    
    # 3. Analysis of all backbone combinations
    bb_comparison = []
    
    for (dataset, technique), group in df.groupby(['Dataset', 'Technique']):
        bb_means = group.groupby('Backbone')['Mean'].mean()
        if len(bb_means) >= 2:
            for bb1, bb2 in combinations(bb_means.index, 2):
                delta = bb_means[bb2] - bb_means[bb1]
                bb_comparison.append({
                    'Dataset': dataset,
                    'Technique': technique,
                    'BB1': bb1,
                    'BB2': bb2,
                    'Delta': delta,
                    'BB1_Mean': bb_means[bb1],
                    'BB2_Mean': bb_means[bb2],
                    'Comparison': f"{bb1} vs {bb2}"
                })
    
    bb_comp_df = pd.DataFrame(bb_comparison)
    
    # 4. Aggregate metrics calculation and transformation for ordering
    tech_stats = tech_comp_df.groupby('Comparison')['Delta'].agg(['mean', 'std', 'count']).reset_index()
    bb_stats = bb_comp_df.groupby('Comparison')['Delta'].agg(['mean', 'std', 'count']).reset_index()
    
    # Function to create display comparison names (always show better method first)
    def create_display_comparison(row):
        a, b = row['Comparison'].split(' vs ')
        if row['mean'] >= 0:
            # Positive mean: a → b means a is better than b
            return f"{a} → {b}", row['mean']
        else:
            # Negative mean: b → a means b is better than a, so we invert the sign
            return f"{b} → {a}", abs(row['mean'])
    
    # Apply the mapping and get adjusted means
    tech_stats[['DisplayComparison', 'AdjustedMean']] = tech_stats.apply(
        lambda row: pd.Series(create_display_comparison(row)), axis=1
    )
    bb_stats[['DisplayComparison', 'AdjustedMean']] = bb_stats.apply(
        lambda row: pd.Series(create_display_comparison(row)), axis=1
    )
    
    # Create mapping from original comparison to display comparison
    tech_mapping = dict(zip(tech_stats['Comparison'], tech_stats['DisplayComparison']))
    bb_mapping = dict(zip(bb_stats['Comparison'], bb_stats['DisplayComparison']))
    
    # Create adjusted delta mapping for the display comparisons
    def adjust_delta_for_display(row, mapping):
        original_comp = row['Comparison']
        display_comp = mapping[original_comp]
        a, b = display_comp.split(' → ')
        original_a, original_b = original_comp.split(' vs ')
        
        # If the display order matches original order, keep delta as is
        if a == original_a and b == original_b:
            return row['Delta']
        # If the display order is inverted, invert the delta
        else:
            return -row['Delta']
    
    # Add display comparison and adjusted delta to original dataframes
    tech_comp_df['DisplayComparison'] = tech_comp_df['Comparison'].map(tech_mapping)
    tech_comp_df['AdjustedDelta'] = tech_comp_df.apply(
        lambda row: adjust_delta_for_display(row, tech_mapping), axis=1
    )
    
    bb_comp_df['DisplayComparison'] = bb_comp_df['Comparison'].map(bb_mapping)
    bb_comp_df['AdjustedDelta'] = bb_comp_df.apply(
        lambda row: adjust_delta_for_display(row, bb_mapping), axis=1
    )
    
    # Sort by adjusted mean in descending order
    tech_sorted_order = tech_stats.sort_values('AdjustedMean', ascending=False)['DisplayComparison'].tolist()
    bb_sorted_order = bb_stats.sort_values('AdjustedMean', ascending=False)['DisplayComparison'].tolist()
    
    # Create mean value annotations for the boxplots
    tech_mean_values = dict(zip(tech_stats['DisplayComparison'], tech_stats['AdjustedMean']))
    bb_mean_values = dict(zip(bb_stats['DisplayComparison'], bb_stats['AdjustedMean']))

    # # 5. CREATE SEPARATE HIGH-QUALITY BOXPLOT FIGURES WITH ADJUSTED DATA
    # # Figure 1: Technique Comparisons Boxplot
    # plt.figure(figsize=(16, 10))
    # ax = plt.gca()
    # sns.boxplot(
    #     data=tech_comp_df, 
    #     x='AdjustedDelta', 
    #     y='DisplayComparison', 
    #     order=tech_sorted_order,
    #     showfliers=False,
    #     linewidth=2.0,
    #     ax=ax
    # )
    
    # # Add mean value annotations
    # for i, comparison in enumerate(tech_sorted_order):
    #     mean_val = tech_mean_values[comparison]
    #     # Position the text at the mean value, slightly offset to the right
    #     ax.text(mean_val + 0.5, i, f'μ={mean_val:.2f}', 
    #             va='center', ha='left', fontsize=12, fontweight='bold',
    #             bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
    
    # plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Zero Gain')
    # # plt.title('Performance Gain Distribution: Technique Comparisons', fontsize=20, fontweight='bold', pad=30)
    # plt.xlabel('Performance Gain (%)', fontsize=18, fontweight='bold')
    # plt.ylabel('Technique Comparison', fontsize=18, fontweight='bold')
    # plt.xticks(fontsize=16, fontweight='bold')
    # plt.yticks(fontsize=14, fontweight='bold')
    # plt.grid(axis='x', alpha=0.3, linestyle='--')
    # plt.legend(fontsize=14)
    
    # # Set x-axis to start from 0 since all values should be positive now
    # # plt.xlim(0, max(tech_comp_df['AdjustedDelta'].max() * 1.1, 10))
    # plt.xlim(-35, 35)
    # plt.tight_layout()
    # plt.savefig('technique_comparisons_boxplot.png', dpi=300, bbox_inches='tight', 
    #             facecolor='white', edgecolor='none')
    # plt.savefig('technique_comparisons_boxplot.pdf', bbox_inches='tight', 
    #             facecolor='white', edgecolor='none')
    # plt.show()
    
    # # Figure 2: Backbone Comparisons Boxplot
    # plt.figure(figsize=(16, 10))
    # ax = plt.gca()
    # sns.boxplot(
    #     data=bb_comp_df, 
    #     x='AdjustedDelta', 
    #     y='DisplayComparison', 
    #     order=bb_sorted_order,
    #     showfliers=False,
    #     linewidth=2.0,
    #     ax=ax
    # )
    
    # # Add mean value annotations
    # for i, comparison in enumerate(bb_sorted_order):
    #     mean_val = bb_mean_values[comparison]
    #     # Position the text at the mean value, slightly offset to the right
    #     ax.text(mean_val + 0.5, i, f'μ={mean_val:.2f}', 
    #             va='center', ha='left', fontsize=12, fontweight='bold',
    #             bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
    
    # plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Zero Gain')
    # # plt.title('Performance Gain Distribution: Backbone Comparisons', fontsize=20, fontweight='bold', pad=30)
    # plt.xlabel('Performance Gain (%)', fontsize=18, fontweight='bold')
    # plt.ylabel('Backbone Comparison', fontsize=18, fontweight='bold')
    # plt.xticks(fontsize=16, fontweight='bold')
    # plt.yticks(fontsize=14, fontweight='bold')
    # plt.grid(axis='x', alpha=0.3, linestyle='--')
    # plt.legend(fontsize=14)
    
    # # Set x-axis to start from 0 since all values should be positive now
    # plt.xlim(-35, 35)
    
    # plt.tight_layout()
    # plt.savefig('backbone_comparisons_boxplot.png', dpi=300, bbox_inches='tight', 
    #             facecolor='white', edgecolor='none')
    # plt.savefig('backbone_comparisons_boxplot.pdf', bbox_inches='tight', 
    #             facecolor='white', edgecolor='none')
    # plt.show()
    # 5. CREATE SEPARATE HIGH-QUALITY BOXPLOT FIGURES WITH ADJUSTED DATA
    # Calculate mean values for colormapping
    tech_mean_values = tech_comp_df.groupby('DisplayComparison')['AdjustedDelta'].mean().loc[tech_sorted_order]
    bb_mean_values = bb_comp_df.groupby('DisplayComparison')['AdjustedDelta'].mean().loc[bb_sorted_order]

    # Create normalization for colormaps
    # tech_norm = plt.Normalize(-35, 35)  # Based on your xlim range
    # bb_norm = plt.Normalize(-35, 35)    # Based on your xlim range
    tech_norm = plt.Normalize(0, 14)  # Based on your xlim range
    bb_norm = plt.Normalize(0, 14)    # Based on your xlim range

    # Create color palettes based on mean values
    tech_colors = [plt.cm.RdYlGn(tech_norm(value)) for value in tech_mean_values.values]
    bb_colors = [plt.cm.RdYlGn(bb_norm(value)) for value in bb_mean_values.values]

    # Figure 1: Technique Comparisons Boxplot with colormapping
    plt.figure(figsize=(16, 10))
    ax = plt.gca()

    # Create boxplot with colored boxes based on mean values
    boxplot1 = sns.boxplot(
        data=tech_comp_df, 
        x='AdjustedDelta', 
        y='DisplayComparison', 
        order=tech_sorted_order,
        showfliers=False,
        linewidth=2.0,
        palette=tech_colors,
        ax=ax
    )

    # Add colorbar
    sm1 = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, norm=tech_norm)
    sm1.set_array([])
    cbar1 = plt.colorbar(sm1, ax=ax, shrink=0.8, pad=0.02)
    cbar1.set_label('Mean Performance Gain (%)', fontsize=14, fontweight='bold')
    cbar1.ax.tick_params(labelsize=12)

    # Add mean value annotations
    for i, comparison in enumerate(tech_sorted_order):
        mean_val = tech_mean_values[comparison]
        mean_std = tech_stats[tech_stats['DisplayComparison'] == comparison]['std'].iloc[0]
        # Position the text at the mean value, slightly offset to the right
        ax.text(mean_val - 25, i, f'μ={mean_val:.2f}, σ={mean_std:.2f}', 
                    va='center', ha='right', fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))

    plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Zero Gain')
    # plt.title('Performance Gain Distribution: Technique Comparisons', fontsize=20, fontweight='bold', pad=30)
    plt.xlabel('Performance Gain (%)', fontsize=18, fontweight='bold')
    plt.ylabel('Technique Comparison', fontsize=18, fontweight='bold')
    plt.xticks(fontsize=16, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.grid(axis='x', alpha=0.3, linestyle='--')
    plt.legend(fontsize=14)

    # Set x-axis limits
    plt.xlim(-35, 35)
    plt.tight_layout()
    plt.savefig('technique_comparisons_boxplot.png', dpi=300, bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    plt.savefig('technique_comparisons_boxplot.pdf', bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    plt.show()

    # Figure 2: Backbone Comparisons Boxplot with colormapping
    plt.figure(figsize=(16, 10))
    ax = plt.gca()

    # Create boxplot with colored boxes based on mean values
    boxplot2 = sns.boxplot(
        data=bb_comp_df, 
        x='AdjustedDelta', 
        y='DisplayComparison', 
        order=bb_sorted_order,
        showfliers=False,
        linewidth=2.0,
        palette=bb_colors,
        ax=ax
    )

    # Add colorbar
    sm2 = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, norm=bb_norm)
    sm2.set_array([])
    cbar2 = plt.colorbar(sm2, ax=ax, shrink=0.8, pad=0.02)
    cbar2.set_label('Mean Performance Gain (%)', fontsize=14, fontweight='bold')
    cbar2.ax.tick_params(labelsize=12)

    # Add mean value annotations
    for i, comparison in enumerate(bb_sorted_order):
        mean_val = bb_mean_values[comparison]
        mean_std = bb_stats[bb_stats['DisplayComparison'] == comparison]['std'].iloc[0]
        # Position the text at the mean value, slightly offset to the right
        # if its the comparison spread to the left we will put to right
        # specifically inserted
        if mean_val.round(2) == 1.72:
            ax.text(mean_val + 15, i, f'μ={mean_val:.2f}, σ={mean_std:.2f}', 
                va='center', ha='left', fontsize=12, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
        else:
            ax.text(mean_val - 20, i, f'μ={mean_val:.2f}, σ={mean_std:.2f}', 
                    va='center', ha='right', fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))

    plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Zero Gain')
    # plt.title('Performance Gain Distribution: Backbone Comparisons', fontsize=20, fontweight='bold', pad=30)
    plt.xlabel('Performance Gain (%)', fontsize=18, fontweight='bold')
    plt.ylabel('Encoder Comparison', fontsize=18, fontweight='bold')
    plt.xticks(fontsize=16, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.grid(axis='x', alpha=0.3, linestyle='--')
    plt.legend(fontsize=14)

    # Set x-axis limits
    plt.xlim(-35, 35)
    plt.tight_layout()
    plt.savefig('backbone_comparisons_boxplot.png', dpi=300, bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    plt.savefig('backbone_comparisons_boxplot.pdf', bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    plt.show()

    
    
    
    return {
        'tech_comparisons': tech_comp_df,
        'bb_comparisons': bb_comp_df,
        # 'tech_stats': tech_stats_display,
        # 'bb_stats': bb_stats_display,
        'tech_mapping': tech_mapping,
        'bb_mapping': bb_mapping
    }

# Execute comprehensive analysis
results = comprehensive_analysis(combined_df)

print("\n boxplots saved as:")
print("   - technique_comparisons_boxplot.png (300 DPI)")
print("   - technique_comparisons_boxplot.pdf (vector format)")
print("   - backbone_comparisons_boxplot.png (300 DPI)")
print("   - backbone_comparisons_boxplot.pdf (vector format)")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy import stats
import plotly.express as px

# Seaborn styling for publication
sns.set(style="whitegrid", font_scale=1.5)
sns.set_context(
    "paper",
    rc={
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.labelweight": "bold",
        "grid.color": "gray",
        "grid.linestyle": "--",
        "grid.linewidth": 1.0,
    },
)

def comprehensive_analysis(df):
    # 1. Data filtering and preparation
    valid_techniques = ['Supervised', 'TFC', 'LFR', 'TNC', 'Diet']
    valid_backbones = ['CNN-PFF', 'TS2Vec Encoder', 'RNN', 'ResNet-1D',"ResNet-SE-5", 'IMU Transformer','TS-TCC Encoder']
    
    df = df.copy()
    df = df[df['Technique'].isin(valid_techniques)]
    df = df[df['Backbone'].isin(valid_backbones)]
    
    df['Technique'] = df['Technique'].replace({'TFC': 'TF-C'})
    # 2. Analysis of all technique combinations
    tech_comparison = []
    
    for (dataset, backbone), group in df.groupby(['Dataset', 'Backbone']):
        tech_means = group.groupby('Technique')['Mean'].mean()
        if len(tech_means) >= 2:
            for tech1, tech2 in combinations(tech_means.index, 2):
                delta = tech_means[tech2] - tech_means[tech1]
                tech_comparison.append({
                    'Dataset': dataset,
                    'Backbone': backbone,
                    'Tech1': tech1,
                    'Tech2': tech2,
                    'Delta': delta,
                    'Tech1_Mean': tech_means[tech1],
                    'Tech2_Mean': tech_means[tech2],
                    'Comparison': f"{tech1} vs {tech2}"
                })
    
    tech_comp_df = pd.DataFrame(tech_comparison)
    
    # 3. Analysis of all backbone combinations
    bb_comparison = []
    
    for (dataset, technique), group in df.groupby(['Dataset', 'Technique']):
        bb_means = group.groupby('Backbone')['Mean'].mean()
        if len(bb_means) >= 2:
            for bb1, bb2 in combinations(bb_means.index, 2):
                delta = bb_means[bb2] - bb_means[bb1]
                bb_comparison.append({
                    'Dataset': dataset,
                    'Technique': technique,
                    'BB1': bb1,
                    'BB2': bb2,
                    'Delta': delta,
                    'BB1_Mean': bb_means[bb1],
                    'BB2_Mean': bb_means[bb2],
                    'Comparison': f"{bb1} vs {bb2}"
                })
    
    bb_comp_df = pd.DataFrame(bb_comparison)
    
    # 4. Aggregate metrics calculation and transformation for ordering
    tech_stats = tech_comp_df.groupby('Comparison')['Delta'].agg(['mean', 'std', 'count']).reset_index()
    bb_stats = bb_comp_df.groupby('Comparison')['Delta'].agg(['mean', 'std', 'count']).reset_index()
    
    # Function to create display comparison names (always show better method first)
    def create_display_comparison(row):
        a, b = row['Comparison'].split(' vs ')
        if row['mean'] >= 0:
            # Positive mean: a → b means a is better than b
            return f"{a} → {b}", row['mean']
        else:
            # Negative mean: b → a means b is better than a, so we invert the sign
            return f"{b} → {a}", abs(row['mean'])
    
    # Apply the mapping and get adjusted means
    tech_stats[['DisplayComparison', 'AdjustedMean']] = tech_stats.apply(
        lambda row: pd.Series(create_display_comparison(row)), axis=1
    )
    bb_stats[['DisplayComparison', 'AdjustedMean']] = bb_stats.apply(
        lambda row: pd.Series(create_display_comparison(row)), axis=1
    )
    
    # Create mapping from original comparison to display comparison
    tech_mapping = dict(zip(tech_stats['Comparison'], tech_stats['DisplayComparison']))
    bb_mapping = dict(zip(bb_stats['Comparison'], bb_stats['DisplayComparison']))
    
    # Create adjusted delta mapping for the display comparisons
    def adjust_delta_for_display(row, mapping):
        original_comp = row['Comparison']
        display_comp = mapping[original_comp]
        a, b = display_comp.split(' → ')
        original_a, original_b = original_comp.split(' vs ')
        
        # If the display order matches original order, keep delta as is
        if a == original_a and b == original_b:
            return row['Delta']
        # If the display order is inverted, invert the delta
        else:
            return -row['Delta']
    
    # Add display comparison and adjusted delta to original dataframes
    tech_comp_df['DisplayComparison'] = tech_comp_df['Comparison'].map(tech_mapping)
    tech_comp_df['AdjustedDelta'] = tech_comp_df.apply(
        lambda row: adjust_delta_for_display(row, tech_mapping), axis=1
    )
    
    bb_comp_df['DisplayComparison'] = bb_comp_df['Comparison'].map(bb_mapping)
    bb_comp_df['AdjustedDelta'] = bb_comp_df.apply(
        lambda row: adjust_delta_for_display(row, bb_mapping), axis=1
    )
    
    # Sort by adjusted mean in descending order
    tech_sorted_order = tech_stats.sort_values('AdjustedMean', ascending=False)['DisplayComparison'].tolist()
    bb_sorted_order = bb_stats.sort_values('AdjustedMean', ascending=False)['DisplayComparison'].tolist()
    
    # Create mean value annotations for the boxplots
    tech_mean_values = dict(zip(tech_stats['DisplayComparison'], tech_stats['AdjustedMean']))
    bb_mean_values = dict(zip(bb_stats['DisplayComparison'], bb_stats['AdjustedMean']))

    # # 5. CREATE SEPARATE HIGH-QUALITY BOXPLOT FIGURES WITH ADJUSTED DATA
    # # Figure 1: Technique Comparisons Boxplot
    # plt.figure(figsize=(16, 10))
    # ax = plt.gca()
    # sns.boxplot(
    #     data=tech_comp_df, 
    #     x='AdjustedDelta', 
    #     y='DisplayComparison', 
    #     order=tech_sorted_order,
    #     showfliers=False,
    #     linewidth=2.0,
    #     ax=ax
    # )
    
    # # Add mean value annotations
    # for i, comparison in enumerate(tech_sorted_order):
    #     mean_val = tech_mean_values[comparison]
    #     # Position the text at the mean value, slightly offset to the right
    #     ax.text(mean_val + 0.5, i, f'μ={mean_val:.2f}', 
    #             va='center', ha='left', fontsize=12, fontweight='bold',
    #             bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
    
    # plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Zero Gain')
    # # plt.title('Performance Gain Distribution: Technique Comparisons', fontsize=20, fontweight='bold', pad=30)
    # plt.xlabel('Performance Gain (%)', fontsize=18, fontweight='bold')
    # plt.ylabel('Technique Comparison', fontsize=18, fontweight='bold')
    # plt.xticks(fontsize=16, fontweight='bold')
    # plt.yticks(fontsize=14, fontweight='bold')
    # plt.grid(axis='x', alpha=0.3, linestyle='--')
    # plt.legend(fontsize=14)
    
    # # Set x-axis to start from 0 since all values should be positive now
    # # plt.xlim(0, max(tech_comp_df['AdjustedDelta'].max() * 1.1, 10))
    # plt.xlim(-35, 35)
    # plt.tight_layout()
    # plt.savefig('technique_comparisons_boxplot.png', dpi=300, bbox_inches='tight', 
    #             facecolor='white', edgecolor='none')
    # plt.savefig('technique_comparisons_boxplot.pdf', bbox_inches='tight', 
    #             facecolor='white', edgecolor='none')
    # plt.show()
    
    # # Figure 2: Backbone Comparisons Boxplot
    # plt.figure(figsize=(16, 10))
    # ax = plt.gca()
    # sns.boxplot(
    #     data=bb_comp_df, 
    #     x='AdjustedDelta', 
    #     y='DisplayComparison', 
    #     order=bb_sorted_order,
    #     showfliers=False,
    #     linewidth=2.0,
    #     ax=ax
    # )
    
    # # Add mean value annotations
    # for i, comparison in enumerate(bb_sorted_order):
    #     mean_val = bb_mean_values[comparison]
    #     # Position the text at the mean value, slightly offset to the right
    #     ax.text(mean_val + 0.5, i, f'μ={mean_val:.2f}', 
    #             va='center', ha='left', fontsize=12, fontweight='bold',
    #             bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
    
    # plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Zero Gain')
    # # plt.title('Performance Gain Distribution: Backbone Comparisons', fontsize=20, fontweight='bold', pad=30)
    # plt.xlabel('Performance Gain (%)', fontsize=18, fontweight='bold')
    # plt.ylabel('Backbone Comparison', fontsize=18, fontweight='bold')
    # plt.xticks(fontsize=16, fontweight='bold')
    # plt.yticks(fontsize=14, fontweight='bold')
    # plt.grid(axis='x', alpha=0.3, linestyle='--')
    # plt.legend(fontsize=14)
    
    # # Set x-axis to start from 0 since all values should be positive now
    # plt.xlim(-35, 35)
    
    # plt.tight_layout()
    # plt.savefig('backbone_comparisons_boxplot.png', dpi=300, bbox_inches='tight', 
    #             facecolor='white', edgecolor='none')
    # plt.savefig('backbone_comparisons_boxplot.pdf', bbox_inches='tight', 
    #             facecolor='white', edgecolor='none')
    # plt.show()
    # 5. CREATE SEPARATE HIGH-QUALITY BOXPLOT FIGURES WITH ADJUSTED DATA
    # Calculate mean values for colormapping
    tech_mean_values = tech_comp_df.groupby('DisplayComparison')['AdjustedDelta'].mean().loc[tech_sorted_order]
    bb_mean_values = bb_comp_df.groupby('DisplayComparison')['AdjustedDelta'].mean().loc[bb_sorted_order]

    # Create normalization for colormaps
    # tech_norm = plt.Normalize(-35, 35)  # Based on your xlim range
    # bb_norm = plt.Normalize(-35, 35)    # Based on your xlim range
    tech_norm = plt.Normalize(0, 14)  # Based on your xlim range
    bb_norm = plt.Normalize(0, 14)    # Based on your xlim range

    # Create color palettes based on mean values
    tech_colors = [plt.cm.RdYlGn(tech_norm(value)) for value in tech_mean_values.values]
    bb_colors = [plt.cm.RdYlGn(bb_norm(value)) for value in bb_mean_values.values]

    # Figure 1: Technique Comparisons Boxplot with colormapping
    plt.figure(figsize=(16, 10))
    ax = plt.gca()

    # Create boxplot with colored boxes based on mean values
    boxplot1 = sns.boxplot(
        data=tech_comp_df, 
        x='AdjustedDelta', 
        y='DisplayComparison', 
        order=tech_sorted_order,
        showfliers=False,
        linewidth=2.0,
        palette=tech_colors,
        ax=ax
    )

    # Add colorbar
    sm1 = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, norm=tech_norm)
    sm1.set_array([])
    cbar1 = plt.colorbar(sm1, ax=ax, shrink=0.8, pad=0.02)
    cbar1.set_label('Mean Performance Gain (%)', fontsize=14, fontweight='bold')
    cbar1.ax.tick_params(labelsize=12)

    # Add mean value annotations
    for i, comparison in enumerate(tech_sorted_order):
        mean_val = tech_mean_values[comparison]
        mean_std = tech_stats[tech_stats['DisplayComparison'] == comparison]['std'].iloc[0]
        # Position the text at the mean value, slightly offset to the right
        ax.text(mean_val - 25, i, f'μ={mean_val:.2f}, σ={mean_std:.2f}', 
                    va='center', ha='right', fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

    plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Zero Gain')
    # plt.title('Performance Gain Distribution: Technique Comparisons', fontsize=20, fontweight='bold', pad=30)
    plt.xlabel('Performance Gain (%)', fontsize=18, fontweight='bold')
    plt.ylabel('Technique Comparison', fontsize=18, fontweight='bold')
    plt.xticks(fontsize=16, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.grid(axis='x', alpha=0.3, linestyle='--')
    plt.legend(fontsize=14)

    # Set x-axis limits
    plt.xlim(-35, 35)
    plt.tight_layout()
    plt.savefig('technique_comparisons_boxplot.png', dpi=300, bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    plt.savefig('technique_comparisons_boxplot.pdf', bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    plt.show()

    # Figure 2: Backbone Comparisons Boxplot with colormapping
    plt.figure(figsize=(16, 10))
    ax = plt.gca()

    # Create boxplot with colored boxes based on mean values
    boxplot2 = sns.boxplot(
        data=bb_comp_df, 
        x='AdjustedDelta', 
        y='DisplayComparison', 
        order=bb_sorted_order,
        showfliers=False,
        linewidth=2.0,
        palette=bb_colors,
        ax=ax
    )

    # Add colorbar
    sm2 = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, norm=bb_norm)
    sm2.set_array([])
    cbar2 = plt.colorbar(sm2, ax=ax, shrink=0.8, pad=0.02)
    cbar2.set_label('Mean Performance Gain (%)', fontsize=14, fontweight='bold')
    cbar2.ax.tick_params(labelsize=12)

    # Add mean value annotations
    for i, comparison in enumerate(bb_sorted_order):
        mean_val = bb_mean_values[comparison]
        mean_std = bb_stats[bb_stats['DisplayComparison'] == comparison]['std'].iloc[0]
        # Position the text at the mean value, slightly offset to the right
        # if its the comparison spread to the left we will put to right
        # specifically inserted
        if mean_val.round(2) == 1.72:
            ax.text(mean_val + 15, i, f'μ={mean_val:.2f}, σ={mean_std:.2f}', 
                va='center', ha='left', fontsize=12, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))
        else:
            ax.text(mean_val - 20, i, f'μ={mean_val:.2f}, σ={mean_std:.2f}', 
                    va='center', ha='right', fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

    plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Zero Gain')
    # plt.title('Performance Gain Distribution: Backbone Comparisons', fontsize=20, fontweight='bold', pad=30)
    plt.xlabel('Performance Gain (%)', fontsize=18, fontweight='bold')
    plt.ylabel('Encoder Comparison', fontsize=18, fontweight='bold')
    plt.xticks(fontsize=16, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.grid(axis='x', alpha=0.3, linestyle='--')
    plt.legend(fontsize=14)

    # Set x-axis limits
    plt.xlim(-35, 35)
    plt.tight_layout()
    plt.savefig('backbone_comparisons_boxplot.png', dpi=300, bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    plt.savefig('backbone_comparisons_boxplot.pdf', bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    plt.show()

    
    
    
    return {
        'tech_comparisons': tech_comp_df,
        'bb_comparisons': bb_comp_df,
        # 'tech_stats': tech_stats_display,
        # 'bb_stats': bb_stats_display,
        'tech_mapping': tech_mapping,
        'bb_mapping': bb_mapping
    }

# Execute comprehensive analysis
results = comprehensive_analysis(combined_df)

print("\n boxplots saved as:")
print("   - technique_comparisons_boxplot.png (300 DPI)")
print("   - technique_comparisons_boxplot.pdf (vector format)")
print("   - backbone_comparisons_boxplot.png (300 DPI)")
print("   - backbone_comparisons_boxplot.pdf (vector format)")